# Fine-Tuning LLMs Using LoRA

This notebook demonstrates how to fine-tune the **Qwen2.5-0.5B** language model on the **MBZUAI/LaMini-instruction** dataset using **LoRA (Low-Rank Adaptation)** — a Parameter Efficient Fine-Tuning (PEFT) technique.

LoRA freezes the pretrained model weights and injects small trainable low-rank matrices into the attention layers, drastically reducing the number of trainable parameters and the memory footprint of fine-tuning.

**References**
- Dataset: https://huggingface.co/datasets/MBZUAI/LaMini-instruction
- Base model: https://huggingface.co/Qwen/Qwen2-0.5B-Instruct
- LLM Leaderboard: https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard
- Article: https://medium.com/@pritisagar0427/fine-tuning-the-large-language-models-llms-using-lora-e0d9cc8960cc

## 1. Install Dependencies

- **transformers** — Hugging Face library for pretrained models and tokenizers
- **peft** — Parameter-Efficient Fine-Tuning utilities (LoRA, prefix tuning, etc.)
- **bitsandbytes** — 8-bit / 4-bit quantization for memory-efficient training
- **datasets** — Hugging Face datasets hub access

In [ ]:
!pip install --upgrade transformers peft bitsandbytes
!pip install -q datasets

## 2. Imports

In [ ]:
from typing import Dict, List
from datasets import Dataset, load_dataset, disable_caching
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from torch.utils.data import Dataset
from IPython.display import Markdown
import torch

# Avoid stale cached datasets when re-running
disable_caching()

## 3. Load the Dataset

We load the `MBZUAI/LaMini-instruction` dataset and select the first 200 examples for a quick fine-tuning demo. Each record contains an `instruction`, a `response`, and an `instruction_source`.

In [ ]:
# Load the train split and take a small subset for demonstration
dataset = load_dataset('MBZUAI/LaMini-instruction', split='train')
small_dataset = dataset.select([i for i in range(200)])

print(small_dataset)
print(small_dataset[0])

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset({
    features: ['instruction', 'response', 'instruction_source'],
    num_rows: 200
})
{'instruction': 'List 5 reasons why someone should learn to code', 'response': '1. High demand for coding skills in the job market\n2. Increased problem-solving and analytical skills\n3. Ability to develop new products and technologies\n4. Potentially higher earning potential\n5. Opportunity to work remotely and/or freelance', 'instruction_source': 'alpaca'}


## 4. Format Prompts

The model is trained on instruction-following text. We wrap each `instruction`/`response` pair into a single `text` field that follows a consistent prompt template, so the model learns to map *Instruction → Response*.

In [ ]:
# Templates used to format every record into instruction-tuning text
prompt_template = (
    "Below is an instruction that describes a task, "
    "Write a response that appropriately complete the request. "
    "Instruction : {instruction} \n Response: "
)
answer_template = "{response}"

def _add_text(rec):
    """Add `prompt`, `answer`, and combined `text` fields to a record."""
    instruction = rec["instruction"]
    response = rec["response"]
    if not instruction:
        raise ValueError(f"Expected an instruction in: {rec}")
    if not response:
        raise ValueError(f"Expected a response in: {rec}")

    rec["prompt"] = prompt_template.format(instruction=instruction)
    rec["answer"] = answer_template.format(response=response)
    rec["text"] = rec["prompt"] + rec["answer"]
    return rec

small_dataset = small_dataset.map(_add_text)
print(small_dataset[0])

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

{'instruction': 'List 5 reasons why someone should learn to code', 'response': '1. High demand for coding skills in the job market\n2. Increased problem-solving and analytical skills\n3. Ability to develop new products and technologies\n4. Potentially higher earning potential\n5. Opportunity to work remotely and/or freelance', 'instruction_source': 'alpaca', 'prompt': 'Below is an instruction that describes a task, Write a response that appropriately complete the request. Instruction : List 5 reasons why someone should learn to code \n Response: ', 'answer': '1. High demand for coding skills in the job market\n2. Increased problem-solving and analytical skills\n3. Ability to develop new products and technologies\n4. Potentially higher earning potential\n5. Opportunity to work remotely and/or freelance', 'text': 'Below is an instruction that describes a task, Write a response that appropriately complete the request. Instruction : List 5 reasons why someone should learn to code \n Respon

## 5. Load the Pretrained Model

We use **Qwen2.5-0.5B** (≈500M parameters) loaded in **8-bit precision** via `bitsandbytes` to reduce GPU memory usage. Larger Qwen variants (e.g., 7B) are split into shards on the Hugging Face Hub and can be swapped in by changing `model_id`.

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B"

# Tokenizer — reuse EOS as PAD since Qwen has no dedicated pad token
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load the model in 8-bit precision and let accelerate place layers on GPU/CPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    load_in_8bit=True,
    torch_dtype=torch.float16,
)

# Make sure embedding matrix matches tokenizer vocab size
model.resize_token_embeddings(len(tokenizer))

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Embedding(151665, 896)

## 6. Tokenize and Split the Dataset

We tokenize every example with a fixed `MAX_LENGTH = 256`, copy `input_ids` into `labels` (standard causal-LM setup), and split off 14 examples for evaluation. A `DataCollatorForSeq2Seq` handles dynamic padding during training.

In [ ]:
from functools import partial
import copy
from transformers import DataCollatorForSeq2Seq

MAX_LENGTH = 256

def _preprocess_batch(batch: Dict[str, List]):
    model_inputs = tokenizer(
        batch["text"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )
    # For causal LM, labels are the input_ids shifted internally by the model
    model_inputs["labels"] = copy.deepcopy(model_inputs["input_ids"])
    return model_inputs

_preprocessing_function = partial(_preprocess_batch)

encoded_dataset = small_dataset.map(
    _preprocessing_function,
    batched=True,
    remove_columns=["instruction", "response", "prompt", "answer"],
)

# Safety filter — drop any record that exceeds MAX_LENGTH
processed_dataset = encoded_dataset.filter(lambda rec: len(rec["input_ids"]) <= MAX_LENGTH)

# Train / test split (14 examples held out)
split_dataset = processed_dataset.train_test_split(test_size=14, seed=42)
print(split_dataset)

# Data collator pads to multiples of 8 for efficient GPU compute
data_collator = DataCollatorForSeq2Seq(
    model=model,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
    pad_to_multiple_of=8,
    padding="max_length",
)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction_source', 'text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 186
    })
    test: Dataset({
        features: ['instruction_source', 'text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14
    })
})


## 7. Configure LoRA

LoRA approximates weight updates with low-rank matrices `A · B`, where only `A` and `B` are trained. Key hyperparameters:

| Parameter | Value | Description |
|---|---|---|
| `r` | 256 | Rank of the low-rank matrices |
| `lora_alpha` | 512 | Scaling factor (effective LR multiplier `alpha / r`) |
| `lora_dropout` | 0.05 | Dropout applied to LoRA layers |
| `target_modules` | `q_proj`, `v_proj` | Attention layers LoRA is injected into |
| `task_type` | `CAUSAL_LM` | Causal language modeling |

`prepare_model_for_kbit_training` patches the 8-bit base model so gradients can flow through the LoRA adapters.

In [ ]:
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

# alpha / r is the effective scaling factor applied to LoRA updates
LORA_R = 256
LORA_ALPHA = 512
LORA_DROPOUT = 0.05

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

# Prepare the 8-bit model for training and wrap it with LoRA adapters
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# Sanity check — only ~3% of parameters should be trainable
model.print_trainable_parameters()

trainable params: 17,301,504 || all params: 511,091,456 || trainable%: 3.3852


## 8. Train the Model

We use Hugging Face `Trainer` with `fp16=True` for mixed-precision training. The full base model stays frozen — only the LoRA adapter weights (`adapter_model.bin`) are updated and saved.

In [ ]:
import os
# Disable Weights & Biases logging for this run
os.environ["WANDB_DISABLED"] = "true"

from transformers import TrainingArguments, Trainer
import bitsandbytes

EPOCHS = 3
LEARNING_RATE = 2e-5
MODEL_SAVE_FOLDER_NAME = "qwen-0.5-lora"

training_args = TrainingArguments(
    output_dir=MODEL_SAVE_FOLDER_NAME,
    overwrite_output_dir=True,
    fp16=True,                      # mixed-precision via bitsandbytes
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    data_collator=data_collator,
)

# Disable KV cache during training (needed during inference)
model.config.use_cache = False
trainer.train()

# Save only the small LoRA adapter weights — much cheaper than the full model
trainer.model.save_pretrained(MODEL_SAVE_FOLDER_NAME)
trainer.save_model(MODEL_SAVE_FOLDER_NAME)
trainer.model.config.save_pretrained(MODEL_SAVE_FOLDER_NAME)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-12-02d1a0a35098>:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torc

Epoch,Training Loss,Validation Loss
1,1.175300,0.315691
2,0.384800,0.312606
3,0.357700,0.314991


/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reen

## 9. Inference with the Fine-Tuned Model

Once training is done, we can generate responses with the LoRA-adapted model. To load a specific checkpoint instead of the in-memory `trainer.model`, point the pipeline at e.g. `qwen-0.5-lora/checkpoint-558`.

In [ ]:
from transformers import pipeline

def postprocess(response: str) -> str:
    """Clean up generated text (strip surrounding whitespace)."""
    return response.strip()

# Prompt for prediction
inference_prompt = "List 5 reasons why someone should learn to code."

# Inference pipeline using the fine-tuned model
inf_pipeline = pipeline(
    "text-generation",
    model=trainer.model,
    tokenizer=tokenizer,
    max_length=256,
    trust_remote_code=True,
)

response = inf_pipeline(inference_prompt)[0]["generated_text"]
formatted_response = postprocess(response)
print(formatted_response)

Device set to use cuda:0
The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'GitForCausalLM', 'GlmForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'GraniteForCausalLM', 'GraniteMoeForCausalLM', 'JambaForCausalLM', 'JetMoeForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'Mamba2ForCausalLM', 'MarianFor

List 5 reasons why someone should learn to code. 1. Career Opportunities: With the increasing demand for coding skills, learning to code can open up a wide range of career opportunities in various industries.

2. Personal Growth: Learning to code can help individuals develop problem-solving, critical thinking, and communication skills that can be applied in various aspects of life.

3. Personal Satisfaction: Coding can be a rewarding and fulfilling activity that can bring personal satisfaction and fulfillment.

4. Access to Technology: With the advent of technology, learning to code can open up new opportunities to access technology and solve problems.

5. Personal Branding: Learning to code can help individuals build a personal brand and showcase their skills to potential employers and colleagues.


## 10. Evaluate with ROUGE and BLEU

We compare the model's generated answer against a reference (ground-truth) answer using two standard text-overlap metrics:

- **ROUGE** — measures recall of n-grams; reported as ROUGE-1, ROUGE-2, and ROUGE-L
- **BLEU** — measures precision of n-grams, with smoothing for short sequences

In [ ]:
!pip install rouge-score nltk

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=9f04948502c3d0da11d76be52dd8c1add422d3c7889cf512cb6e9893b6bb411a
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score


### ROUGE Score

In [ ]:
from rouge_score import rouge_scorer

# Reference (ground truth) and generated text
reference = (
    "1. High demand for coding skills in the job market\n"
    "2. Increased problem-solving and analytical skills\n"
    "3. Ability to develop new products and technologies\n"
    "4. Potentially higher earning potential\n"
    "5."
)
generated = (
    '" 1. Career Opportunities: With the increasing demand for coding skills, learning to code can open up a wide range of career opportunities in various industries. '
    "2. Personal Growth: Learning to code can help individuals develop problem-solving, critical thinking, and communication skills that can be applied in various aspects of life."
    "3. Personal Satisfaction: Coding can be a rewarding and fulfilling activity that can bring personal satisfaction and fulfillment."
    "4. Access to Technology: With the advent of technology, learning to code can open up new opportunities to access technology and solve problems. "
    "5. Personal Branding: Learning to code can help individuals build a personal brand and showcase their skills to potential employers and colleagues. \""
)

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
scores = scorer.score(reference, generated)

print("ROUGE Scores:")
for key, value in scores.items():
    print(f"{key}: Precision: {value.precision:.4f}, Recall: {value.recall:.4f}, F1-Score: {value.fmeasure:.4f}")

ROUGE Scores:
rouge1: Precision: 0.1930, Recall: 0.7097, F1-Score: 0.3034
rouge2: Precision: 0.0354, Recall: 0.1333, F1-Score: 0.0559
rougeL: Precision: 0.1404, Recall: 0.5161, F1-Score: 0.2207


### BLEU Score

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

reference = [
    "High demand for coding skills in the job market. "
    "Increased problem-solving and analytical skills. "
    "Ability to develop new products and technologies. "
    "Potentially higher earning potential.".split()
]
generated = (
    "Career Opportunities: With the increasing demand for coding skills, learning to code can open up a wide range of career opportunities in various industries. "
    "Personal Growth: Learning to code can help individuals develop problem-solving, critical thinking, and communication skills that can be applied in various aspects of life. "
    "Personal Satisfaction: Coding can be a rewarding and fulfilling activity that can bring personal satisfaction and fulfillment. "
    "Access to Technology: With the advent of technology, learning to code can open up new opportunities to access technology and solve problems. "
    "Personal Branding: Learning to code can help individuals build a personal brand and showcase their skills to potential employers and colleagues."
).split()

# Smoothing avoids zero scores when n-gram overlap is sparse
smooth = SmoothingFunction().method4
bleu_score = sentence_bleu(reference, generated, smoothing_function=smooth)

print(f"BLEU Score: {bleu_score:.4f}")

BLEU Score: 0.0168


## Summary

We fine-tuned **Qwen2.5-0.5B** on a 200-sample slice of `MBZUAI/LaMini-instruction` using **LoRA** (`r=256`, `alpha=512`, targeting `q_proj` / `v_proj`). Only **3.39%** of the model parameters (≈17.3M out of 511M) were actually trained, while the rest of the network stayed frozen — making the run fast and memory-friendly even on a single GPU.

A blog write-up of this work is available here: <https://medium.com/@pritisagar0427/fine-tuning-the-large-language-models-llms-using-lora-e0d9cc8960cc>